# GPT-4o Column Detection — Evaluation Notebook

This notebook benchmarks the OpenAI GPT-4o vision model on structural column detection in architectural floor plan PDFs.

**What this notebook does:**
1. Runs GPT-4o on a labelled test PDF (289 human corrections available as ground truth)
2. Evaluates Precision / Recall / F1 using IoU matching against human labels
3. Visualises detections — annotated page, per-tile grid, confidence histogram
4. Benchmarks against prior models in the DB (qwen, moondream, SEA-LION, grid_detector)
5. Estimates API cost per PDF

**Ground-truth signal**: Human corrections stored in `detections.db` — 67 confirmed TPs, 197 rejected FPs, 25 missed TN→TP adds.

In [ ]:
import sys, os, json, sqlite3, math
from pathlib import Path
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Add agent directory to path
AGENT_DIR = Path(".").resolve()
sys.path.insert(0, str(AGENT_DIR))
import agent

print(f"Agent loaded: DEFAULT_MODEL={agent.DEFAULT_MODEL}")
print(f"VALID_SHAPES: {agent.VALID_SHAPES}")
status = agent.get_status()
print(f"API status:   {status}")

## 1 — Configuration

In [ ]:
# ── Test PDFs ─────────────────────────────────────────────────────────────────
# Primary: has 289 human corrections → ground truth for quantitative evaluation
TEST_PDF_GT   = Path("/home/jiezhi/Documents/floor-plan-pdf/2aad5b45-ce32-40a6-a00d-e5b0e67f5d08.pdf")

# Secondary: real floor plans, no ground truth yet → qualitative inspection
TEST_PDFS_QUAL = [
    Path("/home/jiezhi/Documents/floor-plan-pdf/TGCH-TD-S-200-L1-00.pdf"),
    Path("/home/jiezhi/Documents/floor-plan-pdf/TGCH-TD-S-200-L3-00.pdf"),
    Path("/home/jiezhi/Documents/floor-plan-pdf/TGCH-TD-S-200-B1-00.pdf"),
]

# IoU threshold for matching a detection to a ground-truth box
IOU_THRESHOLD = 0.25

# Confidence threshold below which we discard detections
CONF_THRESHOLD = 0.5

print(f"Ground-truth PDF : {TEST_PDF_GT.name}  exists={TEST_PDF_GT.exists()}")
for p in TEST_PDFS_QUAL:
    print(f"Qualitative PDF  : {p.name}  exists={p.exists()}")

## 2 — Load Ground Truth from Human Corrections

The feedback UI stored human corrections in `detections.db`. We reconstruct page-space bounding boxes from the stored tile coordinates.

In [ ]:
def corrections_to_page_coords(file_path_like: str, page_width: int, page_height: int) -> dict:
    """
    Convert stored corrections (tile-normalised bbox) to page-space pixel coords.
    Returns dict with keys 'tp' (confirmed), 'fp' (rejected), 'fn' (added/missed).
    """
    con = agent._db()
    rows = con.execute(
        "SELECT action, tile_index, x_offset, y_offset, "
        "       bbox_x1, bbox_y1, bbox_x2, bbox_y2, notes"
        " FROM corrections"
        " WHERE file_path LIKE ?"
        " ORDER BY tile_index",
        (f"%{file_path_like}%",)
    ).fetchall()
    con.close()

    result = {"tp": [], "fp": [], "fn": []}
    for r in rows:
        x_off, y_off = r["x_offset"], r["y_offset"]
        # tile content width/height (capped at TILE_SIZE, capped at page edge)
        tile_w = min(agent.TILE_SIZE, page_width  - x_off)
        tile_h = min(agent.TILE_SIZE, page_height - y_off)
        # de-normalise: correction bbox is stored as fraction of tile content dims
        px1 = r["bbox_x1"] * tile_w + x_off
        py1 = r["bbox_y1"] * tile_h + y_off
        px2 = r["bbox_x2"] * tile_w + x_off
        py2 = r["bbox_y2"] * tile_h + y_off
        bbox = [px1, py1, px2, py2]
        entry = {"bbox": bbox, "notes": r["notes"] or "",
                 "tile_index": r["tile_index"]}
        if r["action"] == "confirm":
            result["tp"].append(entry)
        elif r["action"] == "reject":
            result["fp"].append(entry)   # human said this WAS a false positive
        else:  # "add" — missed by model, human added
            result["fn"].append(entry)

    return result


# Render the PDF page to get pixel dimensions
gt_img, gt_pages = agent._pdf_to_pil(TEST_PDF_GT, page_num=0, dpi=agent.RENDER_DPI)
PW, PH = gt_img.size
print(f"GT image size: {PW}x{PH} px   (DPI={agent.RENDER_DPI})")

gt = corrections_to_page_coords("2aad5b45", PW, PH)
print(f"Ground-truth labels loaded:")
print(f"  Confirmed TP bboxes : {len(gt['tp'])}")
print(f"  Rejected FP bboxes  : {len(gt['fp'])}  (model false positives)")
print(f"  Missed FN bboxes    : {len(gt['fn'])}  (model missed these)")
print()
print(f"Pseudo-ground-truth total true columns: {len(gt['tp']) + len(gt['fn'])}")

## 3 — Run GPT-4o Detection

In [ ]:
print(f"Running GPT-4o on: {TEST_PDF_GT.name}")
print("This makes real OpenAI API calls — expect ~1-2 min for a single-page A0 drawing.")
print("-" * 60)

result = agent.detect_file(
    TEST_PDF_GT,
    page_num=0,
    model=agent.DEFAULT_MODEL,
    dpi=agent.RENDER_DPI,
    verbose=True,
)

if "error" in result:
    print(f"\nERROR: {result['error']}")
else:
    dets = result["detections"]
    print(f"\nGPT-4o detected   : {result['total_columns']} columns")
    print(f"Tiles processed   : {result['stats']['tiles']} total, {result['stats']['tiles_skipped']} skipped")
    print(f"Avg confidence    : {result['stats']['avg_confidence']:.3f}")
    print(f"Shape breakdown   : {result['stats']['by_shape']}")
    print(f"Run ID            : {result['run_id']}")

## 4 — Quantitative Evaluation: Precision / Recall / F1

We match each detection to the nearest ground-truth box using IoU ≥ threshold.  
Ground truth = confirmed TPs (67) + added FNs (25) = 92 true columns.

In [ ]:
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0:
        return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)


def evaluate(detections, gt_boxes, fp_boxes, iou_thresh=IOU_THRESHOLD, conf_thresh=CONF_THRESHOLD):
    """
    Match detections against GT boxes using greedy IoU matching.
    Returns per-detection labels + summary metrics.
    """
    filtered = [d for d in detections if d.get("confidence", 0) >= conf_thresh]
    gt_bboxes = [g["bbox"] for g in gt_boxes]
    fp_bboxes = [g["bbox"] for g in fp_boxes]

    matched_gt = set()
    det_labels = []   # "TP", "FP_confirmed" (overlaps known FP), "FP_new"

    for det in sorted(filtered, key=lambda d: d["confidence"], reverse=True):
        db = det["bbox_page"]
        # Try to match against a GT box
        best_gt_idx, best_iou = -1, 0.0
        for i, gb in enumerate(gt_bboxes):
            if i in matched_gt:
                continue
            v = iou(db, gb)
            if v > best_iou:
                best_iou, best_gt_idx = v, i
        if best_iou >= iou_thresh:
            matched_gt.add(best_gt_idx)
            det_labels.append((det, "TP", best_iou))
        else:
            # Check if it overlaps a known FP (caught by human correction)
            fp_iou = max((iou(db, fb) for fb in fp_bboxes), default=0.0)
            label = "FP_known" if fp_iou >= iou_thresh else "FP_new"
            det_labels.append((det, label, fp_iou))

    tp_count  = sum(1 for _, l, _ in det_labels if l == "TP")
    fp_count  = len(filtered) - tp_count
    fn_count  = len(gt_bboxes) - tp_count   # GT boxes not matched

    precision = tp_count / len(filtered) if filtered else 0.0
    recall    = tp_count / len(gt_bboxes) if gt_bboxes else 0.0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0

    return {
        "det_labels": det_labels,
        "tp": tp_count, "fp": fp_count, "fn": fn_count,
        "precision": precision, "recall": recall, "f1": f1,
        "n_det": len(filtered), "n_gt": len(gt_bboxes),
    }


# Ground truth = confirmed TPs + human-added missed columns
all_gt_boxes = gt["tp"] + gt["fn"]
metrics = evaluate(dets, all_gt_boxes, gt["fp"])

print("=" * 50)
print(f"GPT-4o Evaluation (IoU≥{IOU_THRESHOLD}, conf≥{CONF_THRESHOLD})")
print("=" * 50)
print(f"  Detections after filter : {metrics['n_det']}")
print(f"  Ground-truth columns    : {metrics['n_gt']}  (67 confirmed + 25 adds)")
print()
print(f"  True  Positives (TP)    : {metrics['tp']}")
print(f"  False Positives (FP)    : {metrics['fp']}")
print(f"  False Negatives (FN)    : {metrics['fn']}")
print()
print(f"  Precision : {metrics['precision']:.3f}  ({metrics['precision']*100:.1f}%)")
print(f"  Recall    : {metrics['recall']:.3f}  ({metrics['recall']*100:.1f}%)")
print(f"  F1 score  : {metrics['f1']:.3f}")
print()
fp_breakdown = Counter(l for _, l, _ in metrics["det_labels"] if l.startswith("FP"))
print(f"  FP breakdown: {dict(fp_breakdown)}")
print(f"    FP_known = overlaps a box humans already rejected (repeat error)")
print(f"    FP_new   = novel false positive not seen before")

## 5 — Sensitivity Analysis: Confidence Threshold vs P/R/F1

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]
rows_pr = []
for t in thresholds:
    m = evaluate(dets, all_gt_boxes, gt["fp"], conf_thresh=t)
    rows_pr.append((t, m["n_det"], m["tp"], m["fp"], m["fn"],
                    m["precision"], m["recall"], m["f1"]))

# Print table
print(f"{'Conf':>5} {'Dets':>5} {'TP':>4} {'FP':>4} {'FN':>4} {'Prec':>6} {'Rec':>6} {'F1':>6}")
print("-" * 50)
for t, n, tp, fp, fn, p, r, f in rows_pr:
    marker = " ←" if abs(t - CONF_THRESHOLD) < 0.01 else ""
    print(f"{t:>5.2f} {n:>5} {tp:>4} {fp:>4} {fn:>4} {p:>6.3f} {r:>6.3f} {f:>6.3f}{marker}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, [r[5] for r in rows_pr], "o-", label="Precision", color="steelblue")
ax.plot(thresholds, [r[6] for r in rows_pr], "s-", label="Recall",    color="darkorange")
ax.plot(thresholds, [r[7] for r in rows_pr], "^-", label="F1",        color="forestgreen", linewidth=2)
ax.axvline(CONF_THRESHOLD, color="gray", linestyle="--", alpha=0.6, label=f"Default threshold={CONF_THRESHOLD}")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Score")
ax.set_title("GPT-4o — Precision / Recall / F1 vs Confidence Threshold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("eval_pr_curve.png", dpi=120)
plt.show()

## 6 — Visualise Detections on Full Page

- **Green** box = True Positive (matched GT)
- **Red** box = False Positive (unmatched)
- **Cyan** circle = Ground-truth column (GT box centre)
- **Orange** circle = Human-added column (was missed by model)

In [ ]:
def draw_eval_overlay(img: Image.Image, det_labels, gt_boxes, fn_boxes, scale=0.25) -> Image.Image:
    out  = img.copy()
    draw = ImageDraw.Draw(out)
    # Detected boxes
    for det, label, _ in det_labels:
        bb = det["bbox_page"]
        colour = (0, 220, 60) if label == "TP" else (220, 40, 40)
        x1, y1, x2, y2 = [int(v) for v in bb]
        draw.rectangle([x1, y1, x2, y2], outline=colour, width=3)
        conf = det.get("confidence", 0)
        draw.text((x1+2, max(0,y1-16)), f"{int(conf*100)}%",
                  fill=colour)
    # GT confirmed boxes (cyan dots)
    for g in gt_boxes:
        bb = g["bbox"]
        cx, cy = int((bb[0]+bb[2])/2), int((bb[1]+bb[3])/2)
        r = 6
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline=(0,230,230), width=3)
    # FN boxes (orange — missed columns)
    for g in fn_boxes:
        bb = g["bbox"]
        x1, y1, x2, y2 = [int(v) for v in bb]
        draw.rectangle([x1, y1, x2, y2], outline=(255,165,0), width=3)
        draw.text((x1+2, max(0,y1-16)), "MISS", fill=(255,165,0))
    # Resize for display
    W, H = out.size
    return out.resize((int(W*scale), int(H*scale)), Image.LANCZOS)


filtered_labels = [(d,l,s) for d,l,s in metrics["det_labels"]]
thumb = draw_eval_overlay(gt_img, filtered_labels, gt["tp"], gt["fn"], scale=0.25)

fig, ax = plt.subplots(figsize=(16, 12))
ax.imshow(thumb)
ax.axis("off")

legend = [
    mpatches.Patch(color=(0,220/255,60/255),   label=f"TP — matched GT ({metrics['tp']})"),
    mpatches.Patch(color=(220/255,40/255,40/255), label=f"FP — unmatched ({metrics['fp']})"),
    mpatches.Patch(facecolor="none", edgecolor=(0,230/255,230/255),
                   linewidth=2, label=f"GT confirmed ({len(gt['tp'])})"),
    mpatches.Patch(color=(1,165/255,0),         label=f"GT missed/FN ({len(gt['fn'])})"),
]
ax.legend(handles=legend, loc="upper right", fontsize=11)
ax.set_title(f"GPT-4o Detections — {TEST_PDF_GT.name}\n"
             f"P={metrics['precision']:.2f}  R={metrics['recall']:.2f}  F1={metrics['f1']:.2f}  "
             f"(IoU≥{IOU_THRESHOLD}, conf≥{CONF_THRESHOLD})",
             fontsize=13)
plt.tight_layout()
plt.savefig("eval_page_overlay.png", dpi=120)
plt.show()

## 7 — Per-Tile Drill-Down

Shows each processed tile (skipped tiles excluded) with its detections labelled.

In [ ]:
def tile_thumbnail(tile_img, tile_dets, size=256):
    """Annotate a tile with its detections and resize."""
    out  = tile_img.copy().resize((size, size), Image.LANCZOS)
    draw = ImageDraw.Draw(out)
    W, H = tile_img.size
    scale = size / max(W, H)
    for det in tile_dets:
        bb = det.get("bbox_tile", [])
        if len(bb) != 4:
            continue
        x1,y1,x2,y2 = [int(v*scale) for v in bb]
        conf  = det.get("confidence", 0)
        col   = (0,200,60) if conf>=0.8 else (220,180,0) if conf>=0.5 else (220,60,60)
        draw.rectangle([x1,y1,x2,y2], outline=col, width=2)
        draw.text((x1+1, max(0,y1-11)), f"{int(conf*100)}%", fill=col)
    return out


# Group detections by tile index
by_tile = defaultdict(list)
for d in dets:
    by_tile[d["tile_index"]].append(d)

# Regenerate tiles (without re-detecting)
tiles = list(agent._tiles(gt_img, page_num=0))

# Build grid of tiles that have at least one detection (or skip for brevity)
active_tiles = [(tile_img, info) for tile_img, info in tiles
                if info.index in by_tile or agent._tile_has_content(tile_img, info)]

print(f"Total tiles: {len(tiles)}  |  Tiles with detections: {len(by_tile)}  |  Tiles with content: {len(active_tiles)}")

THUMB_SIZE = 220
ncols      = 6
nrows      = math.ceil(len(active_tiles) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.2, nrows*3.2))
axes = axes.flatten() if nrows > 1 else axes

for ax_i, (tile_img, info) in enumerate(active_tiles):
    tile_dets = by_tile.get(info.index, [])
    thumb = tile_thumbnail(tile_img, tile_dets, THUMB_SIZE)
    axes[ax_i].imshow(thumb)
    axes[ax_i].set_title(
        f"Tile {info.index}  ({info.x_offset},{info.y_offset})\n"
        f"{len(tile_dets)} det{'s' if len(tile_dets)!=1 else ''}",
        fontsize=7
    )
    axes[ax_i].axis("off")

# Hide unused axes
for ax in axes[len(active_tiles):]:
    ax.axis("off")

fig.suptitle(f"Per-Tile Detections — GPT-4o  ({result['total_columns']} total columns)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("eval_tiles_grid.png", dpi=100, bbox_inches="tight")
plt.show()

## 8 — Confidence Distribution

In [ ]:
tp_confs  = [d["confidence"] for d,l,_ in metrics["det_labels"] if l=="TP"]
fp_confs  = [d["confidence"] for d,l,_ in metrics["det_labels"] if l.startswith("FP")]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bins = np.linspace(0, 1, 21)
axes[0].hist(tp_confs, bins=bins, color="steelblue",   alpha=0.8, label=f"TP ({len(tp_confs)})")
axes[0].hist(fp_confs, bins=bins, color="tomato",      alpha=0.8, label=f"FP ({len(fp_confs)})")
axes[0].axvline(CONF_THRESHOLD, color="black", linestyle="--", label=f"threshold={CONF_THRESHOLD}")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Count")
axes[0].set_title("Confidence Distribution — TP vs FP")
axes[0].legend()
axes[0].grid(alpha=0.3)

# All detection confidences
all_confs = [d["confidence"] for d in dets]
axes[1].hist(all_confs, bins=bins, color="mediumpurple", alpha=0.8)
axes[1].axvline(np.mean(all_confs) if all_confs else 0, color="red",
                linestyle="-", label=f"mean={np.mean(all_confs):.2f}" if all_confs else "")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Count")
axes[1].set_title(f"All GPT-4o Detections ({len(all_confs)}) — Confidence Histogram")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("eval_confidence_hist.png", dpi=120)
plt.show()

if all_confs:
    print(f"Confidence stats: min={min(all_confs):.3f}  max={max(all_confs):.3f}  "
          f"mean={np.mean(all_confs):.3f}  median={np.median(all_confs):.3f}")

## 9 — Model Comparison (All Runs in DB)

In [ ]:
con = agent._db()
all_runs = [dict(r) for r in con.execute(
    "SELECT model, COUNT(*) n_runs, AVG(total_cols) avg_cols, MAX(total_cols) max_cols, "
    "       AVG(c.confidence) avg_conf"
    " FROM runs r LEFT JOIN columns c ON r.run_id=c.run_id"
    " GROUP BY model ORDER BY avg_cols DESC"
).fetchall()]
con.close()

print(f"{'Model':<45} {'Runs':>5} {'AvgCols':>8} {'MaxCols':>8} {'AvgConf':>8}")
print("-" * 80)
for r in all_runs:
    model_short = r["model"][:44] if r["model"] else "unknown"
    avg_conf = f"{r['avg_conf']:.3f}" if r["avg_conf"] else "  N/A"
    print(f"{model_short:<45} {r['n_runs']:>5} {r['avg_cols']:>8.1f} {r['max_cols']:>8} {avg_conf:>8}")

# Add GPT-4o from current run
print()
print(f"→ Current run (GPT-4o): {result['total_columns']} columns, "
      f"conf={result['stats']['avg_confidence']:.3f}, "
      f"P={metrics['precision']:.2f} R={metrics['recall']:.2f} F1={metrics['f1']:.2f}")

# Bar chart
models  = [r["model"][:30] if r["model"] else "unknown" for r in all_runs]
avg_cols = [r["avg_cols"] or 0 for r in all_runs]

fig, ax = plt.subplots(figsize=(10, 4))
colours = ["steelblue" if "gpt" in m else
           "darkorange" if "SEA-LION" in m or "sea-lion" in m or "Gemma" in m else
           "gray" for m in models]
bars = ax.barh(models, avg_cols, color=colours, alpha=0.85)
ax.bar_label(bars, fmt="%.1f", padding=4)
ax.set_xlabel("Average columns detected per run")
ax.set_title("Model Comparison — Average Columns Detected")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("eval_model_comparison.png", dpi=120)
plt.show()

## 10 — Run GPT-4o on Qualitative PDFs (TGCH Floor Plans)

These PDFs have no human corrections yet — this shows raw model output.

In [ ]:
qual_results = {}

for pdf_path in TEST_PDFS_QUAL:
    if not pdf_path.exists():
        print(f"  SKIP (not found): {pdf_path.name}")
        continue
    print(f"\nDetecting: {pdf_path.name} ...")
    r = agent.detect_file(pdf_path, page_num=0, verbose=True)
    if "error" in r:
        print(f"  ERROR: {r['error']}")
    else:
        qual_results[pdf_path.name] = r
        print(f"  → {r['total_columns']} columns  conf={r['stats']['avg_confidence']:.3f}  "
              f"tiles={r['stats']['tiles']} (skipped={r['stats']['tiles_skipped']})")

In [ ]:
# Show annotated thumbnail for each qualitative PDF
n = len(qual_results)
if n == 0:
    print("No qualitative results to show.")
else:
    fig, axes = plt.subplots(1, n, figsize=(8*n, 10))
    if n == 1:
        axes = [axes]

    for ax, (name, r) in zip(axes, qual_results.items()):
        pdf_path = next(p for p in TEST_PDFS_QUAL if p.name == name)
        page_img, _ = agent._pdf_to_pil(pdf_path, page_num=0, dpi=agent.RENDER_DPI)
        annotated   = agent.draw_detections(page_img, r["detections"])
        scale       = min(1.0, 2400 / max(annotated.size))
        thumb       = annotated.resize(
            (int(annotated.width*scale), int(annotated.height*scale)), Image.LANCZOS
        )
        ax.imshow(thumb)
        ax.axis("off")
        ax.set_title(
            f"{name}\n{r['total_columns']} columns | conf={r['stats']['avg_confidence']:.3f}",
            fontsize=11
        )

    plt.tight_layout()
    plt.savefig("eval_qualitative.png", dpi=100, bbox_inches="tight")
    plt.show()

## 11 — API Cost Estimate

In [ ]:
# GPT-4o pricing (as of April 2026 — update if changed)
# https://openai.com/pricing
INPUT_TOKEN_COST  = 2.50 / 1_000_000   # $2.50 per 1M input tokens
OUTPUT_TOKEN_COST = 10.0 / 1_000_000   # $10.00 per 1M output tokens

# OpenAI image token cost:
# - "high" detail image 1024x1024 ≈ 765 tokens
# - "low"  detail image            ≈ 85 tokens (reference images)
TILE_IMAGE_TOKENS = 765    # per tile (high detail, 1024x1024)
REF_IMAGE_TOKENS  = 85     # per reference image (low detail)
SYSTEM_TEXT_TOKENS= 400    # system prompt
USER_TEXT_TOKENS  = 250    # user prompt text
OUTPUT_TOKENS_EST = 300    # typical JSON response per tile

n_refs = len(agent._load_references())
stats  = result["stats"]
n_processed = stats["tiles"] - stats["tiles_skipped"]

tokens_per_tile = (SYSTEM_TEXT_TOKENS + USER_TEXT_TOKENS +
                   TILE_IMAGE_TOKENS + n_refs * REF_IMAGE_TOKENS)
input_tokens    = tokens_per_tile * n_processed
output_tokens   = OUTPUT_TOKENS_EST * n_processed
input_cost      = input_tokens  * INPUT_TOKEN_COST
output_cost     = output_tokens * OUTPUT_TOKEN_COST
total_cost      = input_cost + output_cost

print("=" * 55)
print("API Cost Estimate — GPT-4o")
print("=" * 55)
print(f"  Tiles processed       : {n_processed}  (of {stats['tiles']} total)")
print(f"  Reference images/tile : {n_refs}  (low detail, ~{REF_IMAGE_TOKENS} tokens each)")
print(f"  Tokens/tile (input)   : {tokens_per_tile:,}")
print(f"  Total input tokens    : {input_tokens:,}")
print(f"  Total output tokens   : {output_tokens:,}")
print(f"  Input cost            : ${input_cost:.4f}")
print(f"  Output cost           : ${output_cost:.4f}")
print(f"  Total cost / PDF      : ${total_cost:.4f}")
print()
# Cost for 100 PDFs
print(f"  Cost for 100 PDFs     : ${total_cost*100:.2f}")
print(f"  Cost for 1000 PDFs    : ${total_cost*1000:.2f}")
print()
print("Note: Actual token counts vary. Use OpenAI usage dashboard for exact figures.")
print("Tip: Reduce cost by increasing TILE_STEP (fewer tiles) or lowering RENDER_DPI.")

## 12 — Summary & Interpretation

In [ ]:
print("=" * 65)
print("EVALUATION SUMMARY — GPT-4o vs Prior Models")
print("=" * 65)

print(f"""
Test PDF      : {TEST_PDF_GT.name}
Ground truth  : {len(all_gt_boxes)} true columns  (67 confirmed + 25 missed)
Known FP pool : {len(gt['fp'])} boxes that SEA-LION reported as FP

─── GPT-4o Results ──────────────────────────────────────────
  Detected      : {result['total_columns']}
  Precision     : {metrics['precision']:.3f}  ({metrics['tp']} TP / {metrics['n_det']} dets)
  Recall        : {metrics['recall']:.3f}  ({metrics['tp']} TP / {metrics['n_gt']} GT)
  F1 Score      : {metrics['f1']:.3f}
  Avg confidence: {result['stats']['avg_confidence']:.3f}

─── SEA-LION Baseline (from corrections) ────────────────────
  Confirmed TP  : 67
  FP rejected   : 197
  Missed FN     : 25
  Est. Precision: {67/(67+197):.3f}  (67 TP / 264 detected)
  Est. Recall   : {67/(67+25):.3f}  (67 TP / 92 GT)
  Est. F1       : {2*(67/264)*(67/92)/((67/264)+(67/92)):.3f}

─── Prior models (from DB) ───────────────────────────────────
  qwen3-vl:8b       : 0 detections (model too small)
  moondream:latest  : 0–3 detections (bbox-incapable)
  grid_detector     : 33 columns (CV heuristic, no learning)
""")

if metrics['f1'] > 0.5:
    print("✓ GPT-4o is a substantial improvement over all local models.")
    if metrics['precision'] > 67/264:
        print("✓ Precision improved over SEA-LION (fewer false positives).")
    if metrics['recall'] > 67/92:
        print("✓ Recall improved over SEA-LION (fewer missed columns).")
else:
    print("⚠ F1 below 0.5 — review tile size, DPI, or correction data coverage.")

print()
print("─── Recommendations ─────────────────────────────────────────")
best_thresh = max(rows_pr, key=lambda r: r[7])  # max F1
print(f"  Best conf threshold: {best_thresh[0]:.2f}  (F1={best_thresh[7]:.3f})")
print(f"  Current threshold  : {CONF_THRESHOLD}")
if best_thresh[0] != CONF_THRESHOLD:
    print(f"  → Consider setting CONF_THRESHOLD = {best_thresh[0]:.2f}")
print(f"  Est. cost/PDF: ${total_cost:.4f} — scale to {total_cost*1000:.0f} PDFs = ${total_cost*1000:.2f}")